# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

i used a  Bar Chart to compare population growth percentages across all six continents between 1970 and 2022

Easy comparison: A bar chart makes it easy to compare population growth percentages across different continents.

Clear representation: Each bar represents one continent, and its height represents the percentage of population growth.

Highlighting: Africa is highlighted in yellow to draw attention to the fastest-growing continent.

Sorting: The bars are arranged in descending order, making the comparison easier.

Labels: Percentage values are displayed above each bar for better readability.

In [2]:

# Q1 - Which continent is growing fastest?

# Calculate total population by continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

# Get population in 1970 and 2022
pop_1970 = (
    continent_pop[continent_pop['Year'] == 1970]
    .set_index('Continent')['Population']
)

pop_2022 = (
    continent_pop[continent_pop['Year'] == 2022]
    .set_index('Continent')['Population']
)

# Calculate percentage growth since 1970
growth = pd.DataFrame({
    'Population 1970': pop_1970,
    'Population 2022': pop_2022
})

growth['Growth (%)'] = (
    (growth['Population 2022'] - growth['Population 1970'])
    / growth['Population 1970']
) * 100

growth = growth.sort_values('Growth (%)', ascending=False)

# Display results
display(growth.round(2))

# Identify fastest-growing continent
fastest = growth['Growth (%)'].idxmax()

print(f"Fastest growing continent: {fastest}")
print(f"Population growth: {growth.loc[fastest, 'Growth (%)']:.2f}%")

# Create bar chart
fig = px.bar(
    growth.reset_index(),
    x='Continent',
    y='Growth (%)',
    color='Growth (%)',
    text=growth['Growth (%)'].round(1),
    title='Population Growth by Continent (1970–2022)',
    labels={
        'Growth (%)': 'Population Growth (%)',
        'Continent': 'Continent'
    },
    color_continuous_scale='Viridis'
)

fig.update_traces(textposition='outside')

fig.update_layout(
    template='plotly_white',
    width=1000,
    height=600,
    showlegend=False
)

fig.show()


,Population 1970,Population 2022,Growth (%)
Continent,,,
Africa,365444348,1426730932,290.41
Oceania,19480270,45038554,131.20
South America,192947156,436816608,126.39
Asia,2144906290,4721383274,120.12
North America,315434606,600296136,90.31
Europe,655923991,743147538,13.30


Fastest growing continent: Africa
Population growth: 290.41%


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

We used a Horizontal Bar Chart

Comparison: Makes it easy to compare population increases across 15 countries.
Readability: Country names are easier to read horizontally.
Highlighting: African countries are highlighted in yellow, while other countries are shown in grey.
Ranking: Helps identify countries with the largest population increases.
Data labels: Displays population values next to each bar.

In [3]:

# Q2 - Which countries added the most people?

# Select population data for 1970 and 2022
selected_years = df_long[
    df_long['Year'].isin([1970, 2022])
].copy()

# Convert population to numeric
selected_years['Population'] = pd.to_numeric(
    selected_years['Population'], errors='coerce'
)

# Reshape data to compare both years
country_pop = selected_years.pivot_table(
    index=['Country', 'Continent'],
    columns='Year',
    values='Population',
    aggfunc='sum'
).reset_index()

# Keep countries with data for both years
country_pop = country_pop.dropna(subset=[1970, 2022])

# Calculate additional people between 1970 and 2022
country_pop['People Added'] = (
    country_pop[2022] - country_pop[1970]
)

# Find the top 15 countries
top15 = country_pop.sort_values(
    'People Added', ascending=False
).head(15)

# Count African countries in the top 15
africa_count = (top15['Continent'] == 'Africa').sum()

# Display results
print("TOP 15 COUNTRIES BY POPULATION INCREASE")
display(top15[['Country', 'Continent', 'People Added']].reset_index(drop=True))

print(f"\nNumber of African countries in the top 15: {africa_count}")

# Create highlighting categories
top15['Group'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other continents'
)

# Horizontal bar chart
fig = px.bar(
    top15.sort_values('People Added', ascending=True),
    x='People Added',
    y='Country',
    orientation='h',
    color='Group',
    text='People Added',
    color_discrete_map={
        'Africa': '#F4B400',
        'Other continents': '#AAB7B8'
    },
    title='Top 15 Countries by Population Increase (1970–2022)',
    labels={
        'People Added': 'Additional Population',
        'Country': 'Country',
        'Group': 'Region'
    }
)

fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside'
)

fig.update_layout(
    template='plotly_white',
    width=1100,
    height=750,
    yaxis=dict(autorange='reversed'),
    showlegend=True
)

fig.show()


TOP 15 COUNTRIES BY POPULATION INCREASE


Year,Country,Continent,People Added
0,India,Asia,859671872
1,China,Asia,603352887
2,Pakistan,Asia,176533990
3,Nigeria,Africa,162971948
4,Indonesia,Asia,160272945
5,United States,North America,137961517
6,Brazil,South America,118943623
7,Bangladesh,Asia,103644512
8,Ethiopia,Africa,95071678
9,DR Congo,Africa,78858479



Number of African countries in the top 15: 4


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

i used scatter plot graph 

The scatter plot shows a general positive relationship between land area and population.land area alone does not determine population size. Some countries have high populations despite having relatively small land areas, while some large countries have comparatively lower populations
The logarithmic scales help compare countries with very different land areas and population sizes
The continental colours also help identify geographical differences and unusual observations.

In [4]:
# Q3 - Do bigger countries have more people?

import plotly.express as px

# Select 2022 population data
pop_2022 = df_long[df_long['Year'] == 2022].copy()

# Add land area information
area_data = df[['Country', 'Area']].drop_duplicates()

q3_data = pop_2022.merge(area_data, on='Country')

# Create scatter plot
fig = px.scatter(
    q3_data,
    x='Area',
    y='Population',
    color='Continent',
    hover_name='Country',
    log_x=True,
    log_y=True,
    title='Relationship Between Land Area and Population (2022)',
    labels={
        'Area': 'Land Area (km²)',
        'Population': 'Population (2022)',
        'Continent': 'Continent'
    }
)

fig.update_layout(
    template='plotly_white',
    width=1100,
    height=700
)

fig.show()

# Identify countries with the highest and lowest population density
q3_data['Population Density'] = (
    q3_data['Population'] / q3_data['Area']
)

print("Highest population density:")
display(
    q3_data.nlargest(5, 'Population Density')[
        ['Country', 'Continent', 'Area',
         'Population', 'Population Density']
    ]
)

print("Lowest population density:")
display(
    q3_data.nsmallest(5, 'Population Density')[
        ['Country', 'Continent', 'Area',
         'Population', 'Population Density']
    ]
)

Highest population density:


,Country,Continent,Area,Population,Population Density
119,Macau,Asia,30,695168,23172.266667
134,Monaco,Europe,2,36469,18234.500000
187,Singapore,Asia,710,5975689,8416.463380
89,Hong Kong,Asia,1104,7488865,6783.392210
76,Gibraltar,Europe,6,32649,5441.500000


Lowest population density:


,Country,Continent,Area,Population,Population Density
78,Greenland,North America,2166086,56466,0.026068
64,Falkland Islands,South America,12173,3780,0.310523
230,Western Sahara,Africa,266000,575986,2.165361
135,Mongolia,Asia,1564110,3398366,2.172715
141,Namibia,Africa,825615,2567012,3.109212


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

Box plot with individual data points (strip/scatter points).

I selected a box plot with individual data points to compare population density distributions across continents. The box plot displays the median, quartiles and variation, while individual points represent countries and highlight outliers. I used a logarithmic Y-axis because population density varies considerably between countries. Different colours help distinguish the continents and make comparisons easier.

In [7]:

# Q4 - How crowded are the continents?

import pandas as pd
import plotly.express as px

# Select population data for 2022
pop_2022 = df_long[df_long['Year'] == 2022].copy()

# Get country area information
area_data = df[['Country', 'Area']].drop_duplicates()

# Merge population and area data
q4_data = pop_2022.merge(area_data, on='Country')

# Calculate population density
q4_data['Density'] = (
    q4_data['Population'] / q4_data['Area']
)

# Remove invalid values
q4_data = q4_data[
    (q4_data['Area'] > 0) &
    (q4_data['Density'] > 0)
]

# Create box plot
fig = px.box(
    q4_data,
    x='Continent',
    y='Density',
    color='Continent',
    points='all',
    title='Population Density Distribution by Continent (2022)',
    labels={
        'Continent': 'Continent',
        'Density': 'Population Density (people per km²)'
    },
    hover_name='Country'
)

fig.update_layout(
    template='plotly_white',
    width=1100,
    height=700,
    showlegend=False
)

fig.update_yaxes(type='log')

fig.show()

# Summary statistics
print("Population Density Summary by Continent:")

display(
    q4_data.groupby('Continent')['Density']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(2)
)


Population Density Summary by Continent:


,count,median,mean,min,max
Continent,,,,,
Africa,57,59.71,125.05,2.17,871.93
Asia,50,122.97,1025.02,2.17,23172.27
Europe,50,106.87,663.32,3.62,18234.50
North America,40,221.42,272.76,0.03,1299.26
Oceania,23,78.25,132.54,3.40,603.24
South America,14,17.79,20.97,0.31,65.02


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.
I selected a Choropleth Map to visualize population percentage

I selected a Choropleth Map to visualize population percentage changes across countries between 2000 and 2022. This chart is suitable because it displays geographical patterns and makes it easy to identify countries experiencing population growth or decline. I used a red-yellow-green colour scale, where green represents population growth, yellow represents smaller changes, and red represents population decline. The interactive map also allows users to explore individual countries and compare their population changes.

In [8]:

# Q5 - Where did populations grow and where did they shrink?

# Extract population data for 2000 and 2022
pop_2000 = df_long[df_long['Year'] == 2000][
    ['CCA3', 'Country', 'Continent', 'Population']
].rename(columns={'Population': 'Population_2000'})

pop_2022 = df_long[df_long['Year'] == 2022][
    ['CCA3', 'Country', 'Continent', 'Population']
].rename(columns={'Population': 'Population_2022'})

# Merge both years
growth = pd.merge(
    pop_2000,
    pop_2022,
    on=['CCA3', 'Country', 'Continent']
)

# Calculate percentage population change
growth['Percentage_Change'] = (
    (growth['Population_2022'] - growth['Population_2000'])
    / growth['Population_2000']
) * 100

# Sort countries by percentage change
growth = growth.sort_values(
    'Percentage_Change', ascending=False
)

# Display countries with the highest growth
print("TOP 10 COUNTRIES WITH POPULATION GROWTH")
display(growth[['Country', 'Continent',
                'Percentage_Change']].head(10))

# Display countries with the largest decline
print("TOP 10 COUNTRIES WITH POPULATION DECLINE")
display(growth[['Country', 'Continent',
                'Percentage_Change']].tail(10))

# Create a world choropleth map
fig = px.choropleth(
    growth,
    locations='CCA3',
    color='Percentage_Change',
    hover_name='Country',
    hover_data={
        'Continent': True,
        'Percentage_Change': ':.2f',
        'CCA3': False
    },
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title='Population Percentage Change by Country (2000–2022)',
    labels={'Percentage_Change': 'Population Change (%)'}
)

fig.update_layout(
    width=1100,
    height=650,
    geo=dict(showframe=False, showcoastlines=True)
)

fig.show()


TOP 10 COUNTRIES WITH POPULATION GROWTH


,Country,Continent,Percentage_Change
167,Qatar,Asia,317.242239
219,United Arab Emirates,Asia,188.249439
59,Equatorial Guinea,Africa,144.520327
215,Turks and Caicos Islands,North America,143.827358
148,Niger,Africa,125.490255
104,Jordan,Asia,123.209664
108,Kuwait,Asia,120.624880
5,Angola,Africa,117.084619
39,Chad,Africa,114.590398
230,Western Sahara,Africa,113.032270


TOP 10 COUNTRIES WITH POPULATION DECLINE


,Country,Continent,Percentage_Change
218,Ukraine,Europe,-18.776723
229,Wallis and Futuna,Oceania,-21.401888
226,Vatican City,Europe,-21.658986
25,Bosnia and Herzegovina,Europe,-22.630888
111,Latvia,Europe,-22.648786
133,Moldova,Europe,-23.016822
126,Marshall Islands,Oceania,-23.338374
117,Lithuania,Europe,-23.601880
3,American Samoa,Oceania,-23.968745
153,Northern Mariana Islands,Oceania,-38.321840


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.
I selected a Treemap chart

I selected a Treemap chart to visualize the distribution of the world's population across continents and individual countries in 2022. This chart is suitable because it represents hierarchical data in a single figure. The larger rectangles represent continents, while the smaller rectangles represent the countries within them. The size of each rectangle corresponds to its population. Different colours help distinguish continents and make comparisons easier. This visualization helps identify which continents and countries have the largest populations.

In [9]:

# Q6 - Where do the world's 8 billion people live?

# Filter population data for 2022
pop_2022 = df_long[df_long['Year'] == 2022].copy()

# Ensure population is numeric
pop_2022['Population'] = pd.to_numeric(
    pop_2022['Population'], errors='coerce'
)

# Remove missing or invalid values
pop_2022 = pop_2022.dropna(
    subset=['Continent', 'Country', 'Population']
)

pop_2022 = pop_2022[pop_2022['Population'] > 0]

# Create Treemap
fig = px.treemap(
    pop_2022,
    path=['Continent', 'Country'],
    values='Population',
    color='Continent',
    title="World Population Distribution by Continent and Country (2022)",
    color_discrete_sequence=px.colors.qualitative.Set3
)

# Improve chart design
fig.update_layout(
    width=1200,
    height=750,
    title_x=0.5,
    margin=dict(t=80, l=20, r=20, b=20)
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        '<extra></extra>'
    )
)

fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I selected a 100% stacked area chart to visualize changes in the percentage share of world population across different regions between 1970 and 2022. This chart is suitable because it shows how each region contributes to the total world population over time. I grouped North America, South America and Oceania into Rest of the world, as required. Different colours distinguish the four regions, while the continuous areas make it easy to identify increasing and decreasing population shares.

In [10]:

# Q7 - Whose share of the world is rising?

# Group the remaining continents
df_q7 = df_long.copy()

df_q7['Region'] = df_q7['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

# Calculate population by year and region
region_population = (
    df_q7.groupby(['Year', 'Region'])['Population']
    .sum()
    .reset_index()
)

# Calculate total world population for each year
world_population = (
    region_population.groupby('Year')['Population']
    .sum()
    .reset_index(name='World Population')
)

# Merge total population
region_population = region_population.merge(
    world_population,
    on='Year'
)

# Calculate percentage share
region_population['Share (%)'] = (
    region_population['Population']
    / region_population['World Population']
) * 100

# Create stacked area chart
fig = px.area(
    region_population,
    x='Year',
    y='Share (%)',
    color='Region',
    title='Share of World Population by Region (1970–2022)',
    groupnorm='fraction',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(
    width=1100,
    height=650,
    title_x=0.5,
    xaxis_title='Year',
    yaxis_title='Share of World Population (%)',
    yaxis=dict(tickformat='.0%'),
    legend_title='Region'
)

fig.show()
